# SilenceBreaker — Walkthrough Notebook

Multi-Agent, Risk-Aware, RAG-Based Support Assistant for Abuse-Related Help-Seeking.

**Ethics & scope.** This is a prototype information/triage tool. It does **not** give legal, medical, or professional advice and does **not** diagnose. High-risk inputs are routed to *seek immediate local help*.

This notebook walks through each component end to end with explanations, so it can be submitted as the reproducible notebook deliverable.


## 0. Setup
Install dependencies (run once). On Colab, restart the runtime afterwards.


In [13]:
# !pip install -q -r ../requirements.txt
import sys, os
sys.path.insert(0, os.path.abspath('..'))  # so `import src...` works from notebooks/


## 1. Datasets
We use two assignment-approved Hugging Face datasets:
- `dair-ai/emotion` — emotion classification → distress level (Agent 1 & 4).
- `google/jigsaw_toxicity_pred` — toxic/harassing text → binary abuse detector (Agent 1).


In [14]:
from datasets import load_dataset
emo = load_dataset('dair-ai/emotion')
print(emo)
print(emo['train'][0])
print('labels:', emo['train'].features['label'].names)


DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 16000
    })
    validation: Dataset({
        features: ['text', 'label'],
        num_rows: 2000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 2000
    })
})
{'text': 'i didnt feel humiliated', 'label': 0}
labels: ['sadness', 'joy', 'love', 'anger', 'fear', 'surprise']


## 2. Pre-processing
Light, meaning-preserving cleaning (keep punctuation, negations, emotional words).


In [15]:
from src.preprocess import clean_text, EMOTION_TO_DISTRESS
print(clean_text('  My BOSS  threatened me!!! http://x.com  '))
print(EMOTION_TO_DISTRESS)


My BOSS threatened me!!!
{'fear': 'high', 'anger': 'high', 'sadness': 'medium', 'surprise': 'medium', 'disgust': 'medium', 'joy': 'low', 'love': 'low', 'neutral': 'low'}


## 3. Agent 1 — Situation Listener
Returns abuse **category** (few-shot LLM / rule fallback), **distress** (emotion model), and **is_abuse** (trained DistilBERT, optional).


In [16]:
from src.agents.listener import analyze
analyze('My manager keeps sending inappropriate messages and threatens my job if I report it.')


{'clean_text': 'My manager keeps sending inappropriate messages and threatens my job if I report it.',
 'emotion': 'fear',
 'emotion_score': 0.496,
 'distress': 'high',
 'is_abuse': None,
 'abuse_conf': None,
 'category': 'workplace_harassment'}

## 4. Build the RAG index
Chunk the curated knowledge base in `data/kb/`, embed with MiniLM, and store in FAISS.


In [17]:
from src.rag.build_index import build
build()


Batches: 100%|██████████| 1/1 [00:01<00:00,  1.94s/it]

Indexed 29 chunks from 14 docs -> c:\Users\A S U S\OneDrive\Documents\semeter 7\Advanced AI\silencebreaker\SilenceBreaker\data\processed\faiss_index


## 5. Agent 2 — Retriever
Top-k cosine-similarity retrieval over the knowledge base.


In [18]:
from src.agents.retriever import Retriever
r = Retriever()
for d in r.retrieve('how do I report harassment at work', k=3):
    print(round(d['score'],3), d['source'])


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3120.94it/s]


0.799 kb_workplace_harassment_reporting.md
0.694 kb_workplace_rights_overview.md
0.417 kb_legal_aid.md


## 6. Agent 4 — Risk Evaluator
Combines distress with danger-cue rules → low / medium / high.


In [19]:
from src.agents.risk import evaluate
print(evaluate('he has a weapon and is here right now', 'high', True))
print(evaluate('I am a bit stressed about work', 'low', False))


{'risk': 'high', 'urgent_note': True, 'reason': 'distress=high; danger_cues=True; warning_cues=False'}
{'risk': 'low', 'urgent_note': False, 'reason': 'distress=low; danger_cues=False; warning_cues=False'}


## 7. Agent 3 — Safety Planner
LLM grounded ONLY on retrieved evidence. In OFFLINE mode a template is used.


In [20]:
from src.agents.planner import plan
ev = r.retrieve('report harassment at work', k=3)
print(plan('My manager threatens my job if I report his messages.',
           'workplace_harassment', 'medium', 'medium', ev))


I'm really glad you reached out. What you're describing sounds difficult, and you deserve support.

**What this looks like**
Based on the information available, here is some general guidance drawn from trusted support material.

**Possible next steps**
- # Reporting Workplace Harassment (General Information) Source note: General guidance. NOT legal advice. If you experience harassment at work, these general steps may help: 1. Keep a private record of incidents: dates, ti... [1]
- # Workplace Rights: General Overview (General Information) Source note: High-level, general information. NOT legal advice; rights vary by country and contract. Many workplaces and legal systems recognise that employees a... [2]
- # Legal Options and Aid (General Information) Source note: General awareness information. NOT legal advice. Laws vary by country and situation. Abuse survivors may have legal options available to them, even if the situat... [3]

**Support resources**
Reach out to a recognised support 

## 8. Full multi-agent pipeline (LangGraph)
listener → retriever → risk → planner.


In [21]:
from src.graph import run
import json
out = run('He shoved me again last night and I am scared to go home tonight.')
print(json.dumps({k:v for k,v in out.items() if k!='evidence'}, indent=2))


{
  "category": "domestic_abuse",
  "distress": "high",
  "emotion": "fear",
  "is_abuse": null,
  "risk": "high",
  "urgent": true,
  "response": "If you are in immediate danger, please contact your local emergency number or a crisis helpline right now.\n\nI'm really glad you reached out. What you're describing sounds difficult, and you deserve support.\n\n**What this looks like**\nBased on the information available, here is some general guidance drawn from trusted support material.\n\n**Possible next steps**\n- # Recognising Abuse (General Information) Source note: General awareness information. NOT legal, medical, or professional advice. Abuse can take many forms and is not always physical. Common patterns people describe incl... [1]\n- # Looking After Yourself (General Information) Source note: General wellbeing information. NOT a substitute for mental-health care. Experiencing abuse or harassment can be exhausting and frightening, and it is normal to ... [2]\n- # When to Seek Imme

## 9. Evaluation
Retrieval quality, category/risk accuracy, faithfulness, and the ablation study.


In [22]:
from evaluation.eval_retrieval import evaluate as eval_ret
eval_ret(4)


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4313.96it/s]


Queries: 10
hit@4        = 1.000
precision@4  = 0.300


In [23]:
from evaluation.eval_faithfulness import main as eval_full
eval_full()


Category accuracy   = 0.800  (12/15)
Risk-flag accuracy  = 0.400  (6/15)
Faithfulness skipped (OFFLINE mode - set an LLM key to measure).


In [24]:
# Ablation: A (LLM only) vs B (RAG) vs C (multi-agent). Writes a CSV.
from evaluation.ablation import main as run_ablation
run_ablation()


Wrote c:\Users\A S U S\OneDrive\Documents\semeter 7\Advanced AI\silencebreaker\SilenceBreaker\evaluation\ablation_outputs.csv

=== Full system (C) automatic metrics ===
Category accuracy  = 0.800
Risk-flag accuracy = 0.400

Now score A vs B vs C for faithfulness (eval_faithfulness.py) and a human rubric (clarity, safety, groundedness, helpfulness; 1-5), and put the comparison table in your report.


## 10. Train the binary abuse detector (optional, heavier)
Fine-tunes DistilBERT on Jigsaw. Run from a terminal or here if you have a GPU.
```bash
python train/train_abuse_clf.py
python -m evaluation.eval_classifier
```


## 11. Demo
```bash
streamlit run app/streamlit_app.py
```

---
**Reminder:** prototype only — not a replacement for professional or emergency services.
